In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

## loading data

In [ ]:
#specifing datapath
Data_path=Path(r"C:\\Users\\User\\Downloads\\Video Game Sales dataset\\Video_Games_Sales_as_at_22_Dec_2016.csv")
# checking if file exit in the path location
if not Data_path.exists():
    raise FileNotFoundError("CSV not found in the given file path  please check the file path ")
#load the dataset
df=pd.read_csv(Data_path)

In [ ]:
#shape of the dataset
print("shape:",df.shape)

In [ ]:
#prints summary of coln
display(df.info())

In [ ]:
#generate descriptive statistics summary
display(df.describe())

In [ ]:
df.head()

## Checking out for missing values and duplicates

In [ ]:
missing_details=pd.DataFrame({
'missing_count':df.isnull().sum(),
'missing_percentage':(df.isnull().mean()*100).round(2)
}
).sort_values('missing_percentage',ascending=True)
display(missing_details[missing_details['missing_count']>0])

In [ ]:
#checking
missing = df.isnull().sum()

missing = missing[missing > 0]

missing.plot(kind='bar', figsize=(9, 5))

plt.title('Missing Values by Column')
plt.ylabel('Count')
plt.show()

#looking for distribution of global_sales by year

In [ ]:
df.groupby("Year_of_Release")["Global_Sales"].sum().plot(kind="line", marker="o", figsize=(9, 4))
plt.title("Total global sales by release year")
plt.xlabel("Year")
plt.ylabel("Sales (millions)")
plt.show()

In [ ]:
print("Duplicate rows:", df.duplicated().sum())

Since no duplicate rows, so no need to find and remove duplicate rows

##  Data cleaning and validation

In [ ]:
numeric_coln=df.select_dtypes('number').columns.tolist()
print(numeric_coln)

In [ ]:
for i in numeric_coln:
    if i in df.columns:
        df[i]=pd.to_numeric(df[i],errors='coerce') #errors='coerce' means the data that cant be converted will be missing value

In [ ]:
df.select_dtypes('number').columns

In [ ]:
df = df.dropna(subset=['Year_of_Release'])

In [ ]:
df[['User_Score', 'Critic_Score']].isna().sum()

In [ ]:
# user_score range is 0-10 anything above or less is invalid
#replacing them with nan
df.loc[
    ~df.User_Score.between(0,10),'User_Score'
]=np.nan

## Critic_score range is 0-100 anything above or less is invalid
df.loc[
    ~df.Critic_Score.between(0,100),'Critic_Score'
]=np.nan

In [ ]:
df[['User_Score', 'Critic_Score']].isna().sum()

In [ ]:
df[['Critic_Count','User_Count']].isna().sum()

In [ ]:
#count cant be negative so checking and replacing it as nan if its negative
for i in ['Critic_Count','User_Count']:
    if i in df.columns:
        df.loc[df[i]<0,i]=np.nan

In [ ]:
sales_coln=['NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales', 'Global_Sales']
for col in sales_coln:
    if col in df.columns:
        df.loc[df[col]<0,col]=np.nan

In [ ]:
df.Year_of_Release

In [ ]:
# we remove rows having missing global sales value as its our target
if 'Global_Sales' not in df.columns:#check if column present in dataset
    raise KeyError('no Global_Sales column in the dataset')
rows_before=len(df)
df=df.dropna(subset=['Global_Sales']).copy()
print(f"Rows removed from the dataset without Global_Sales value:{rows_before-len(df)}")

In [ ]:
df.Name.shape

In [ ]:
#dropping rows with missing Name
if 'Name' in df.columns:
    rows_before=len(df)
    df=df.dropna(subset=['Name']).copy()
    print(f"Rows removed without game name:{rows_before-len(df)}")

In [ ]:
missing_details=pd.DataFrame({
'missing_count':df.isnull().sum(),
'missing_percentage':(df.isnull().mean()*100).round(2)
}
).sort_values('missing_percentage',ascending=True)
display(missing_details[missing_details['missing_count']>0])

In [ ]:
if "Year_of_Release" in df.columns:
    print("\nRelease-year summary:")
    print(df["Year_of_Release"].describe())

    print("\nMissing release years:",
          df["Year_of_Release"].isna().sum())

In [ ]:
missing_details=pd.DataFrame({
'missing_count':df.isnull().sum(),
'missing_percentage':(df.isnull().mean()*100).round(2)
}
).sort_values('missing_percentage',ascending=True)
display(missing_details[missing_details['missing_count']>0])

## check check

In [ ]:
plt.figure(figsize=(10, 5))

df.isnull().sum().sort_values(
    ascending=False
).plot(kind='bar')

plt.title('Missing Values in Video Game Sales Dataset')
plt.xlabel('Columns')
plt.ylabel('Number of Missing Values')
plt.xticks(rotation=45)

plt.show()

Load the dataset.



Explore the dataset.



Check missing values.



Visualize the original data.



Check skewness of Global_Sales.



Handle missing values.



Split the data into training and testing sets (or split before fitting imputers).



Encode categorical features.



Train the model.

In [ ]:
df.groupby("Year_of_Release")["Global_Sales"].sum().plot(kind="line", marker="o", figsize=(9, 4))
plt.title("Total global sales by release year")
plt.xlabel("Year")
plt.ylabel("Sales (millions)")
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
df.groupby("Genre")["Global_Sales"].sum().sort_values().plot(kind="barh", ax=ax[0], title="Sales by genre")
df.groupby("Platform")["Global_Sales"].sum().sort_values(ascending=False).head(12).plot(kind="bar", ax=ax[1], title="Top 12 platforms")
plt.tight_layout()
plt.show()

In [ ]:
df.head()

In [ ]:
df.Rating.value_counts().plot(kind="bar", figsize=(9, 4))

In [ ]:
print("Duplicate rows:", df.duplicated().sum())
print("Missing target values:", df["Global_Sales"].isna().sum())
print("Release year range:", df["Year_of_Release"].min(), "->", df["Year_of_Release"].max())

# tidy duplicate rating labels and keep the year as a whole number
df["Rating"] = df["Rating"].replace({"K-A": "E", "RP": "Unknown"})
df["Year_of_Release"] = df["Year_of_Release"].astype("Int64")

## EDA

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df["Global_Sales"], bins=60, ax=ax[0]).set_title("Global sales")
##log transormation to reduces the influence of very large values and makes the distribution easier to visualize.
sns.histplot(np.log1p(df["Global_Sales"]), bins=60, ax=ax[1]).set_title("log(1 + global sales)")
plt.tight_layout()
plt.show()
print("Skewness:", round(df["Global_Sales"].skew(), 2))

In [ ]:
df["Global_Sales_log"] = np.log1p(df["Global_Sales"])

print("Original:", df["Global_Sales"].skew())
print("Transformed:", df["Global_Sales_log"].skew())

Most games still have relatively low sales.
A few games have considerably higher sales.


log transformation has reduced skewness from 17.38 to 2.76 which Approximately 84.1% reduction in skewness ,so we found that  it imporves the distribution even though it hasnt made it symmetric.

In [ ]:
df.groupby("Year_of_Release")["Global_Sales"].sum().plot(
    kind="line",
    marker="o",
    figsize=(9, 4)
)

plt.title("Total global sales by release year")
plt.xlabel("Year")
plt.ylabel("Sales (millions)")
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
df.groupby("Genre")["Global_Sales"].sum().sort_values().plot(kind="barh", ax=ax[0], title="Sales by genre")
df.groupby("Platform")["Global_Sales"].sum().sort_values(ascending=False).head(12).plot(kind="bar", ax=ax[1], title="Top 12 platforms")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].scatter(df["Critic_Score"], df["Global_Sales"], s=6, alpha=0.3)
ax[0].set(xlabel="Critic score", ylabel="Global sales", title="Critic score vs sales")
ax[1].scatter(df["User_Score"], df["Global_Sales"], s=6, alpha=0.3)
ax[1].set(xlabel="User score", ylabel="Global sales", title="User score vs sales")
plt.tight_layout()
plt.show()

In [ ]:
num_cols = ["Year_of_Release", "NA_Sales", "EU_Sales", "JP_Sales", "Other_Sales",
            "Global_Sales", "Critic_Score", "Critic_Count", "User_Score", "User_Count"]
plt.figure(figsize=(8, 6))
sns.heatmap(df[num_cols].corr(), cmap="coolwarm", center=0)
plt.title("Correlation between numeric columns")
plt.show()

# Global_Sales is the sum of the four regional columns, so they leak the target
regional = df[["NA_Sales", "EU_Sales", "JP_Sales", "Other_Sales"]].sum(axis=1)
print("Share of rows where the regional sales rebuild Global_Sales:",
      round((regional.round(2) == df["Global_Sales"].round(2)).mean(), 3))

## Feature engineering

In [ ]:
reference_year=df["Year_of_Release"].max()
reference_year

In [ ]:
df["Game_Age"] =reference_year - df["Year_of_Release"]
df["Has_Review_Scores"] = (
    df["Critic_Score"].notna() | df["User_Score"].notna()
).astype(int)
df["Review_Total_Count"] = df["Critic_Count"].fillna(0) + df["User_Count"].fillna(0)
df["Critic_User_Gap"] = df["User_Score"] * 10 - df["Critic_Score"] #bringing user score to the same scale as critic score by multiplying it by 10
df["Is_Major_Publisher"] = df["Publisher"].fillna("").str.contains(
    "Nintendo|Sony|Microsoft|Electronic Arts|Activision|Ubisoft", case=False
).astype(int)

df[["Game_Age", "Has_Review_Scores", "Review_Total_Count", "Critic_User_Gap", "Is_Major_Publisher"]].head()

In [ ]:
platform_groups = {
    # Nintendo
    "Wii": "Nintendo",
    "NES": "Nintendo",
    "GB": "Nintendo",
    "DS": "Nintendo",
    "SNES": "Nintendo",
    "GBA": "Nintendo",
    "3DS": "Nintendo",
    "N64": "Nintendo",
    "WiiU": "Nintendo",
    "GC": "Nintendo",

    # PlayStation
    "PS": "PlayStation",
    "PS2": "PlayStation",
    "PS3": "PlayStation",
    "PS4": "PlayStation",
    "PSP": "PlayStation",
    "PSV": "PlayStation",

    # Xbox
    "X360": "Xbox",
    "XB": "Xbox",
    "XOne": "Xbox",

    # PC
    "PC": "PC",

    # Sega
    "GEN": "Sega",
    "DC": "Sega",
    "SAT": "Sega",
    "SCD": "Sega",
    "GG": "Sega",

    # Atari
    "2600": "Atari",

    # Other platforms
    "WS": "Other",
    "NG": "Other",
    "TG16": "Other",
    "3DO": "Other",
    "PCFX": "Other"
}

df["Platform_Group"] = df["Platform"].map(platform_groups).fillna("Unknown")

In [ ]:
print(df["Platform_Group"].value_counts())

In [ ]:
import matplotlib.pyplot as plt

df["Platform_Group"].value_counts().plot(
    kind="bar",
    figsize=(9, 5)
)

plt.title("Distribution of Gaming Platforms")
plt.xlabel("Platform Group")
plt.ylabel("Number of Games")
plt.xticks(rotation=45)
plt.show()

In [ ]:
df.columns

In [ ]:
df.head(1)

## Delete where more than 2 features are not present

In [ ]:
categorical_features = [

    "Rating",
    "Publisher",
    "Developer"
]
df[categorical_features].isnull().sum(axis=1).gt(2).sum()

In [ ]:
df = df[df[categorical_features].isnull().sum(axis=1) <= 2].copy()

In [ ]:
print((df[categorical_features].isnull().sum(axis=1) > 2).sum())

## Fill missing values (before the train-test split)

In [ ]:
from sklearn.impute import SimpleImputer

# Features available in the selected post-review prediction scenario.
numeric_features = [
    "Game_Age",
    "Has_Review_Scores",
    "Review_Total_Count",
    "Critic_User_Gap",
    "Critic_Score",
    "Critic_Count",
    "User_Score",
    "User_Count",
    "Is_Major_Publisher"
]

categorical_features = [
    "Platform",
    "Genre",
    "Rating",
    "Publisher",
    "Developer"
]

features = numeric_features + categorical_features
target = "Global_Sales"

# All missing values are filled here, on the FULL dataset, BEFORE the split.
# 1. Handle missing numerical values
num_imputer = SimpleImputer(strategy="median")
df[numeric_features] = num_imputer.fit_transform(df[numeric_features])

# 2. Handle missing categorical values
cat_imputer = SimpleImputer(strategy="most_frequent")
df[categorical_features] = cat_imputer.fit_transform(df[categorical_features])

print("Missing values left after filling:", int(df[features].isnull().sum().sum()))

## Train test split

In [ ]:
df.columns.value_counts()

In [ ]:
from sklearn.model_selection import train_test_split

X = df[features].copy()
y = df[target].copy()

# The data has already been imputed, so the split now happens on complete data.
# Random holdout for a comparable baseline benchmark.
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])
print("Training features:", X_train.shape[1])

In [ ]:
df.isnull().sum()

In [ ]:
from sklearn.preprocessing import OneHotEncoder
import pandas as pd
import numpy as np

# Missing values were already filled before the split, so only the encoding of
# the categorical features is left here.

# 1. Encode categorical features (encoder fit on the training split only)
encoder = OneHotEncoder(
    handle_unknown="ignore",
    min_frequency=10,
    sparse_output=False
)

X_train_encoded = encoder.fit_transform(X_train[categorical_features])
X_test_encoded = encoder.transform(X_test[categorical_features])

# 2. Combine numerical and encoded categorical features
encoded_columns = encoder.get_feature_names_out(categorical_features)

X_train_processed = pd.concat([
    X_train[numeric_features].reset_index(drop=True),
    pd.DataFrame(X_train_encoded, columns=encoded_columns)
], axis=1)

X_test_processed = pd.concat([
    X_test[numeric_features].reset_index(drop=True),
    pd.DataFrame(X_test_encoded, columns=encoded_columns)
], axis=1)

# 3. Log-transform target
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

print("Training shape:", X_train_processed.shape)
print("Testing shape:", X_test_processed.shape)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(random_state=42),
    "XGBoost": XGBRegressor(random_state=42)
}

trained_models = {}

for name, model in models.items():
    model.fit(X_train_processed, y_train_log)
    trained_models[name] = model

In [ ]:
# models = {
#     "Linear Regression": LinearRegression(),
#     "Decision Tree": DecisionTreeRegressor(random_state=42),
#     "Random Forest": RandomForestRegressor(
#         n_estimators=300,
#         min_samples_split=5,
#         min_samples_leaf=2,
#         random_state=42,
#         n_jobs=-1
#     ),
#     "XGBoost": XGBRegressor(
#             n_estimators=400,
#             max_depth=6,
#             learning_rate=0.05,
#             subsample=0.8,
#             colsample_bytree=0.8,
#             objective="reg:squarederror",
#             random_state=42,
#             n_jobs=-1
#         )
# }
# trained_models = {}

# for name, model in models.items():
#     model.fit(X_train_processed, y_train_log)
#     trained_models[name] = model

# print("Models trained:", list(trained_models.keys()))

In [ ]:
results = []

for name, model in trained_models.items():
    predictions_log = model.predict(X_test_processed)
    predictions = np.maximum(np.expm1(predictions_log), 0)

    results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, predictions),
        "RMSE": np.sqrt(mean_squared_error(y_test, predictions)),
        "R2": r2_score(y_test, predictions)
    })

results_df = pd.DataFrame(results).sort_values("MAE")
display(results_df)

best_initial_name = results_df.iloc[0]["Model"]
print("Best initial model:", best_initial_name)

In [ ]:
# Select the best-performing model from the comparison table.
best_initial_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_initial_name]

print("Best model based on test MAE:", best_initial_name)

In [ ]:
best_predictions_log = best_model.predict(X_test_processed)
best_predictions = np.maximum(np.expm1(best_predictions_log), 0)

best_result = pd.DataFrame([{
    "Model": best_initial_name,
    "MAE": mean_absolute_error(y_test, best_predictions),
    "RMSE": np.sqrt(mean_squared_error(y_test, best_predictions)),
    "R2": r2_score(y_test, best_predictions)
}])

display(best_result)

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(y_test, best_predictions, alpha=0.35)
limit = max(float(y_test.max()), float(best_predictions.max()))
plt.plot([0, limit], [0, limit], linestyle="--")
plt.xlabel("Actual Global Sales (millions)")
plt.ylabel("Predicted Global Sales (millions)")
plt.title(f"Actual vs Predicted Global Sales — {best_initial_name}")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    best_model,
    X_test_processed,
    y_test_log,
    scoring="neg_mean_absolute_error",
    n_repeats=5,
    random_state=42,
    n_jobs=-1
)

importance_df = pd.DataFrame({
    "Feature": X_test_processed.columns,
    "Importance": perm.importances_mean,
    "Importance_Std": perm.importances_std
}).sort_values("Importance", ascending=False)

display(importance_df.head(20))

plt.figure(figsize=(9, 6))
top_features = importance_df.head(20).sort_values("Importance")
plt.barh(top_features["Feature"], top_features["Importance"])
plt.xlabel("Decrease in model score when feature is shuffled")
plt.title(f"Permutation Importance — {best_initial_name}")
plt.tight_layout()
plt.show()

In [ ]:
xgb_model = XGBRegressor(
    objective="reg:squarederror",
    random_state=42,
    n_jobs=1
)

In [ ]:
param_grid = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [3, 4, 5, 6, 8],
    "learning_rate": [0.01, 0.03, 0.05, 0.1, 0.2],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "min_child_weight": [1, 3, 5, 7],
    "gamma": [0, 0.1, 0.2, 0.3],
    "reg_alpha": [0, 0.01, 0.1, 1],
    "reg_lambda": [1, 2, 5, 10]
}

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
random_search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_grid,
    n_iter=30,
    scoring="neg_root_mean_squared_error",
    cv=5,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train_processed, y_train)

In [ ]:
print("Best Parameters:")
print(random_search.best_params_)

print("\nBest Cross-Validation RMSE:")
print(-random_search.best_score_)

In [ ]:
best_xgb_model = random_search.best_estimator_

y_pred_tuned = best_xgb_model.predict(X_test_processed)

mae = mean_absolute_error(y_test, y_pred_tuned)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_tuned))
r2 = r2_score(y_test, y_pred_tuned)

print("Tuned XGBoost Performance")
print("-------------------------")
print("MAE:", mae)
print("RMSE:", rmse)
print("R2 Score:", r2)

In [ ]:
# Calculate before-tuning model performance
y_pred_before = np.maximum(np.expm1(best_model.predict(X_test_processed)), 0)

before_mae = mean_absolute_error(y_test, y_pred_before)
before_rmse = np.sqrt(mean_squared_error(y_test, y_pred_before))
before_r2 = r2_score(y_test, y_pred_before)

# Tuned model performance is already calculated
# mae, rmse, r2

# Compare both models
comparison = pd.DataFrame({
    "Model": ["XGBoost Before Tuning", "XGBoost After Tuning"],
    "MAE": [before_mae, mae],
    "RMSE": [before_rmse, rmse],
    "R2": [before_r2, r2]
})

display(comparison)

In [ ]:
if rmse < before_rmse and r2 > before_r2:
    final_model = best_xgb_model
    selected_model = "XGBoost After Tuning"
else:
    final_model = best_model
    selected_model = "XGBoost Before Tuning"

print("Selected Model:", selected_model)

import joblib

joblib.dump(final_model, "best_xgboost_model.pkl")

print("Model saved successfully!")

In [ ]:
# Save processed test features
pd.DataFrame(
    X_test_processed
).to_csv("X_test_processed.csv", index=False)

# Save actual test target
y_test.to_csv("y_test.csv", index=False)

print("Test datasets saved successfully!")

In [ ]:
df.to_csv("video_game_sales_cleaned.csv", index=False)

print("Dataset saved successfully!")

In [ ]:
import joblib

joblib.dump(final_model, "best_xgboost_model.pkl")

In [ ]:
import numpy as np
import pandas as pd

# Convert actual test values back to original scale
y_actual = np.expm1(y_test_log)

comparison_results = {}

for name, model in trained_models.items():

    # Predict on log scale
    y_pred_log = model.predict(X_test_processed)

    # Convert predictions back to original scale
    y_pred = np.expm1(y_pred_log)

    comparison_results[name] = y_pred

# Create comparison table
comparison_df = pd.DataFrame({
    "Actual Global Sales": y_actual
})

for name, predictions in comparison_results.items():
    comparison_df[f"{name} Predicted"] = predictions

display(comparison_df)